# 03 — Custom Checks & Cleansing

**Цель ноутбука:** добавить проверки, которые не выражаются стандартными expectations (cross-table FK, conditional checks), и построить cleansing-пайплайн, который реально исправляет найденные проблемы.

**Что делаем:**
1. Custom checks — условные проверки и проверки между таблицами
2. Cleansing pipeline — функции, которые приводят данные в порядок
3. Apply & save — применяем очистку, сохраняем в `data/processed/`
4. Re-validate — прогоняем тот же Checkpoint на очищенных данных, сравниваем
5. Summary — метрики «до и после», которые пойдут в README

**Что важно понять про разделение:**
- Standard expectations (день 2) — **формальные правила**, которые умеет GE
- Custom checks (этот ноутбук) — **бизнес-правила и многотабличная логика**, которые GE 1.x не покрывает напрямую
- Cleansing — **исправление**, а не валидация. Принимает решение «что делать с найденным»

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import great_expectations as gx

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import os
os.chdir(PROJECT_ROOT)
print(f"Working from: {PROJECT_ROOT}")

# Загружаем таблицы
DATA_DIR = Path('data/raw')
TABLE_FILES = {
    'customers': 'olist_customers_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
}
tables = {name: pd.read_csv(DATA_DIR / filename) for name, filename in TABLE_FILES.items()}
for name, df in tables.items():
    print(f"  {name:20} {df.shape[0]:>8,} x {df.shape[1]}")

## 1. Custom checks — условные и многотабличные

Сделаем функцию-обёртку с единым форматом результата. Так все наши custom checks будут давать однородный отчёт, который потом легко агрегировать.

In [ ]:
from dataclasses import dataclass

@dataclass
class CheckResult:
    name: str
    total_checked: int
    violations: int
    success: bool
    sample_violations: list  # примеры для отчёта
    
    @property
    def success_rate(self):
        if self.total_checked == 0:
            return 1.0
        return 1 - self.violations / self.total_checked
    
    def __repr__(self):
        status = "✅" if self.success else "❌"
        return (f"{status} {self.name}: {self.violations:,} violations "
                f"of {self.total_checked:,} ({self.success_rate*100:.2f}% pass)")

### 1.1 Conditional check: статус delivered ⟺ дата доставки

Бизнес-правило: если `order_status == 'delivered'`, то `order_delivered_customer_date` обязательно. И наоборот: если дата проставлена, статус должен быть consistent.

In [ ]:
def check_conditional_not_null(df, condition_col, condition_value, target_col, name=None):
    """Когда condition_col == condition_value, target_col не должен быть null."""
    name = name or f"{target_col} not null when {condition_col}={condition_value}"
    subset = df[df[condition_col] == condition_value]
    null_mask = subset[target_col].isnull()
    violations = null_mask.sum()
    sample = subset[null_mask].head(3).to_dict('records') if violations > 0 else []
    return CheckResult(
        name=name,
        total_checked=len(subset),
        violations=int(violations),
        success=violations == 0,
        sample_violations=sample,
    )

# Применяем
r1 = check_conditional_not_null(
    tables['orders'], 'order_status', 'delivered', 'order_delivered_customer_date'
)
print(r1)

r2 = check_conditional_not_null(
    tables['orders'], 'order_status', 'delivered', 'order_delivered_carrier_date'
)
print(r2)

### 1.2 Date order check: даты в логическом порядке

Из вашего каталога #11 и #12 — нарушения хронологии между датами в `orders`.

In [ ]:
def check_date_order(df, earlier_col, later_col, name=None):
    """earlier_col должен быть <= later_col (NaN игнорируем)."""
    name = name or f"{earlier_col} <= {later_col}"
    earlier = pd.to_datetime(df[earlier_col], errors='coerce')
    later = pd.to_datetime(df[later_col], errors='coerce')
    both_present = earlier.notna() & later.notna()
    violations_mask = both_present & (earlier > later)
    violations = violations_mask.sum()
    sample = df[violations_mask][[earlier_col, later_col]].head(3).to_dict('records')
    return CheckResult(
        name=name,
        total_checked=int(both_present.sum()),
        violations=int(violations),
        success=violations == 0,
        sample_violations=sample,
    )

orders = tables['orders']
r3 = check_date_order(orders, 'order_purchase_timestamp', 'order_approved_at')
r4 = check_date_order(orders, 'order_approved_at', 'order_delivered_carrier_date')
r5 = check_date_order(orders, 'order_delivered_carrier_date', 'order_delivered_customer_date')
for r in [r3, r4, r5]:
    print(r)

### 1.3 Foreign Key integrity (cross-table)

GE 1.x не имеет родного cross-table FK. Пишем сами.

In [ ]:
def check_fk_integrity(child_df, child_col, parent_df, parent_col, name=None):
    """Все значения child_col должны существовать в parent_col."""
    name = name or f"FK {child_col} -> {parent_col}"
    parent_set = set(parent_df[parent_col].dropna().unique())
    child_values = child_df[child_col].dropna()
    orphan_mask = ~child_values.isin(parent_set)
    violations = int(orphan_mask.sum())
    sample = child_values[orphan_mask].head(3).tolist()
    return CheckResult(
        name=name,
        total_checked=len(child_values),
        violations=violations,
        success=violations == 0,
        sample_violations=sample,
    )

r6 = check_fk_integrity(tables['orders'], 'customer_id', tables['customers'], 'customer_id')
r7 = check_fk_integrity(tables['order_items'], 'order_id', tables['orders'], 'order_id')
r8 = check_fk_integrity(tables['order_items'], 'product_id', tables['products'], 'product_id')
r9 = check_fk_integrity(tables['order_items'], 'seller_id', tables['sellers'], 'seller_id')
for r in [r6, r7, r8, r9]:
    print(r)

### 1.4 Empty orders check (#15 из каталога)

Заказы без записей в `order_items` — нельзя считать в GMV.

In [ ]:
def check_orders_have_items(orders_df, order_items_df):
    orders_with_items = set(order_items_df['order_id'].unique())
    all_orders = orders_df['order_id']
    no_items_mask = ~all_orders.isin(orders_with_items)
    violations = int(no_items_mask.sum())
    sample = all_orders[no_items_mask].head(3).tolist()
    return CheckResult(
        name='orders have at least one item',
        total_checked=len(all_orders),
        violations=violations,
        success=violations == 0,
        sample_violations=sample,
    )

r10 = check_orders_have_items(tables['orders'], tables['order_items'])
print(r10)

### 1.5 Все custom-результаты в одну таблицу

In [ ]:
all_custom_results = [r1, r2, r3, r4, r5, r6, r7, r8, r9, r10]

custom_report = pd.DataFrame([
    {
        'check': r.name,
        'total': r.total_checked,
        'violations': r.violations,
        'pass_rate_%': round(r.success_rate * 100, 2),
        'status': '✅' if r.success else '❌',
    }
    for r in all_custom_results
])
custom_report

## 2. Cleansing pipeline

Принципы:
1. **Не модифицируем raw данные** — все cleansing-функции возвращают копию
2. **Каждая функция логирует, что сделала** — кол-во затронутых строк
3. **Решение «что делать с грязью» — явное**: dropna vs fillna vs flag — мы выбираем стратегию для каждой проблемы

### 2.1 Type coercion

In [ ]:
def coerce_orders_types(df: pd.DataFrame) -> pd.DataFrame:
    """Приводим даты к datetime, сохраняем оригинал в copy."""
    df = df.copy()
    date_cols = [
        'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_carrier_date', 'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

def coerce_zip_codes(df: pd.DataFrame, col: str = 'customer_zip_code_prefix') -> pd.DataFrame:
    """ZIP-коды должны быть строкой (с ведущими нулями), не числом."""
    df = df.copy()
    df[col] = df[col].astype(str).str.zfill(5)
    return df

### 2.2 Deduplication

In [ ]:
def dedupe_geolocation(df: pd.DataFrame) -> pd.DataFrame:
    """Полные дубли строк удаляем, оставляя первое вхождение."""
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"  geolocation dedup: {before:,} -> {len(df):,} (-{before - len(df):,})")
    return df

def dedupe_reviews(df: pd.DataFrame, strategy: str = 'keep_first') -> pd.DataFrame:
    """Дубли review_id. Стратегии:
    - keep_first: оставить первое вхождение
    - keep_latest: оставить запись с самой свежей датой
    """
    before = len(df)
    if strategy == 'keep_first':
        df = df.drop_duplicates(subset='review_id', keep='first').reset_index(drop=True)
    elif strategy == 'keep_latest':
        df = df.copy()
        df['review_creation_date'] = pd.to_datetime(df['review_creation_date'])
        df = (df.sort_values('review_creation_date', ascending=False)
                .drop_duplicates(subset='review_id', keep='first')
                .reset_index(drop=True))
    print(f"  reviews dedup ({strategy}): {before:,} -> {len(df):,} (-{before - len(df):,})")
    return df

### 2.3 Null handling

In [ ]:
def fill_category_name(df: pd.DataFrame) -> pd.DataFrame:
    """product_category_name: null -> 'unknown'."""
    df = df.copy()
    before = df['product_category_name'].isna().sum()
    df['product_category_name'] = df['product_category_name'].fillna('unknown')
    print(f"  category_name filled: {before:,} -> 0")
    return df

### 2.4 Filter invalid records

Решение: для критичных нарушений (цена 0, нулевые габариты, payment_value 0) — **фильтруем**. Альтернатива — флаг `is_valid`, оставить решение downstream. Для портфолио проще: фильтруем, но в логах показываем сколько отбросили.

In [ ]:
def filter_invalid_prices(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    before = len(df)
    df = df[df['price'] > 0].reset_index(drop=True)
    print(f"  order_items price > 0: {before:,} -> {len(df):,} (-{before - len(df):,})")
    return df

def filter_invalid_payments(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    before = len(df)
    df = df[(df['payment_type'] != 'not_defined') & (df['payment_value'] > 0)].reset_index(drop=True)
    print(f"  payments valid: {before:,} -> {len(df):,} (-{before - len(df):,})")
    return df

def filter_invalid_product_dimensions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    before = len(df)
    cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
    mask = (df[cols] > 0).all(axis=1) | df[cols].isna().any(axis=1)
    # Решение: если все 4 колонки положительные ИЛИ полностью отсутствуют — оставляем
    df = df[mask].reset_index(drop=True)
    print(f"  products dimensions valid: {before:,} -> {len(df):,} (-{before - len(df):,})")
    return df

## 3. Применяем cleansing и сохраняем

Запускаем весь pipeline и сохраняем результат в `data/processed/`.

In [ ]:
PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

print("Cleansing pipeline:")
print("="*60)

clean = {}
clean['orders'] = coerce_orders_types(tables['orders'])
clean['customers'] = coerce_zip_codes(tables['customers'], 'customer_zip_code_prefix')
clean['sellers'] = coerce_zip_codes(tables['sellers'], 'seller_zip_code_prefix')

clean['geolocation'] = dedupe_geolocation(tables['geolocation'])
clean['order_reviews'] = dedupe_reviews(tables['order_reviews'], strategy='keep_first')

clean['products'] = fill_category_name(tables['products'])
clean['products'] = filter_invalid_product_dimensions(clean['products'])

clean['order_items'] = filter_invalid_prices(tables['order_items'])
clean['order_payments'] = filter_invalid_payments(tables['order_payments'])

# Остальные таблицы — без изменений
for name in ['customers', 'sellers']:
    if name not in clean:
        clean[name] = tables[name]

print("="*60)
print("\nСохраняем в data/processed/:")
for name, df in clean.items():
    path = PROCESSED_DIR / f"{name}_clean.csv"
    df.to_csv(path, index=False)
    print(f"  {path} ({len(df):,} rows)")

## 4. Метрики «до и после»

Самая важная часть для README. Показываем количественный эффект cleansing.

In [ ]:
metrics = []
for name in TABLE_FILES.keys():
    before_rows = len(tables[name])
    after_rows = len(clean[name]) if name in clean else before_rows
    metrics.append({
        'table': name,
        'rows_before': before_rows,
        'rows_after': after_rows,
        'rows_removed': before_rows - after_rows,
        'removed_pct': round((before_rows - after_rows) / before_rows * 100, 2) if before_rows else 0,
    })

metrics_df = pd.DataFrame(metrics)
metrics_df

In [ ]:
# Прогоняем все custom checks на очищенных данных
clean_results = []

# Conditional + date order — на orders
clean_results.append(check_conditional_not_null(
    clean['orders'], 'order_status', 'delivered', 'order_delivered_customer_date'
))

clean['orders']['order_purchase_timestamp'] = pd.to_datetime(clean['orders']['order_purchase_timestamp'])
clean['orders']['order_approved_at'] = pd.to_datetime(clean['orders']['order_approved_at'])
clean['orders']['order_delivered_carrier_date'] = pd.to_datetime(clean['orders']['order_delivered_carrier_date'])
clean['orders']['order_delivered_customer_date'] = pd.to_datetime(clean['orders']['order_delivered_customer_date'])

clean_results.append(check_date_order(
    clean['orders'], 'order_purchase_timestamp', 'order_approved_at'
))

# FK
clean_results.append(check_fk_integrity(
    clean['order_items'], 'order_id', clean['orders'], 'order_id'
))

# Compare
print("Custom checks: ДО vs ПОСЛЕ")
print("="*70)
for orig, after in zip(
    [r1, r3, r7],
    clean_results,
):
    delta = orig.violations - after.violations
    print(f"  {orig.name[:50]:50}")
    print(f"     before: {orig.violations:>6,}  ->  after: {after.violations:>6,}  (Δ {delta:+,})")

**Важное наблюдение:** часть нарушений мы НЕ должны были исправить. Например, conditional-проверка по `delivered_customer_date` останется с нарушениями, если в данных есть orders со статусом `delivered` без даты — это **баг источника**, не наша задача его маскировать. Это и есть «честный» аудит: мы фиксируем то, что не можем безопасно исправить, и сигнализируем upstream.

## 5. Summary metrics для README

Финальный блок цифр, который попадёт в README в раздел «Results».

In [ ]:
# Общая статистика
total_rows_before = sum(len(df) for df in tables.values())
total_rows_after = sum(len(df) for df in clean.values())

total_violations_before = sum(r.violations for r in all_custom_results)
total_violations_after = sum(r.violations for r in clean_results)

print("="*60)
print("DATA AUDIT PIPELINE — SUMMARY")
print("="*60)
print(f"Tables checked:           {len(TABLE_FILES)}")
print(f"Total rows analyzed:      {total_rows_before:,}")
print(f"Expectations (GE):        ~30 (5 suites)")
print(f"Custom checks:            {len(all_custom_results)}")
print(f"")
print(f"Issues found:             {total_violations_before:,}")
print(f"Issues after cleansing:   {total_violations_after:,}")
print(f"Cleansing impact:         -{total_violations_before - total_violations_after:,} issues")
print(f"")
print(f"Rows after cleansing:     {total_rows_after:,} (was {total_rows_before:,})")
print(f"Data Quality Score:       {(1 - total_violations_after/total_rows_after)*100:.2f}%")

## 6. Что дальше (день 4)

К концу дня 3 у вас:
- ✅ 10 custom checks (conditional, date order, cross-table FK, empty orders)
- ✅ Cleansing pipeline с явными стратегиями
- ✅ Очищенные данные в `data/processed/`
- ✅ Метрики «до/после»

**День 4 — финальная упаковка:**
1. Вынести cleansing-функции из ноутбука в `src/cleanser.py`
2. Вынести custom checks в `src/custom_checks.py`
3. Написать pytest-тесты для cleansing-функций (3-5 тестов)
4. Написать **README** — с hero, Quickstart, скриншотами Data Docs, таблицей метрик
5. Опционально: Streamlit-дашборд, Dockerfile, GitHub Actions CI